# Supervised Fine-Tuning with LoRA (Alpaca Format)

Companion notebook for the [Instruction Tuning](https://codemia.ai/courses/llm_inside_and_out/instruction_tuning) lesson on Codemia.

**What you will build**: format a small dataset in Alpaca's `instruction / input / output` structure, wrap `distilgpt2` with LoRA adapters from the `peft` library, and run supervised fine-tuning. Before and after training, sample the model on a held-out set of instructions and compare outputs side-by-side to measure the quality delta.

The previous notebook (`lora_from_scratch.ipynb`) built LoRA by hand. This one uses `peft` so you see what the production workflow actually looks like.

**Time**: ~20-30 minutes on T4

**GPU required**: yes

## Setup

In [ ]:
%pip install -q torch transformers peft datasets matplotlib

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Concept recap

Instruction tuning teaches a base language model to follow instructions instead of merely continuing text. You collect `(instruction, optional input, desired output)` triples and train the model to produce the output given the formatted prompt. The Alpaca format is one popular convention:

```
Below is an instruction that describes a task...
### Instruction:
{instruction}
### Input:
{input}
### Response:
{output}
```

The model is supervised only on the tokens *after* `### Response:` -- the prompt is conditioning, not a target. Typically we use LoRA so the base model's language ability is preserved and we can host many instruction-tuned variants on top of one base.

## Data: a hand-crafted Alpaca-format dataset

In [ ]:
# A tiny curated set that is small enough to train in minutes but large enough
# to move the model toward instruction-following behavior. Real Alpaca is 52k rows.
examples = [
    dict(instruction="Translate the following English to French.", input="Hello, how are you?", output="Bonjour, comment allez-vous?"),
    dict(instruction="Translate the following English to French.", input="Good morning.", output="Bonjour."),
    dict(instruction="Translate the following English to French.", input="Thank you very much.", output="Merci beaucoup."),
    dict(instruction="Translate the following English to French.", input="Where is the train station?", output="Ou est la gare?"),
    dict(instruction="Translate the following English to French.", input="I love reading books.", output="J'aime lire des livres."),
    dict(instruction="What is the capital of the given country?", input="France", output="The capital of France is Paris."),
    dict(instruction="What is the capital of the given country?", input="Japan", output="The capital of Japan is Tokyo."),
    dict(instruction="What is the capital of the given country?", input="Brazil", output="The capital of Brazil is Brasilia."),
    dict(instruction="What is the capital of the given country?", input="Australia", output="The capital of Australia is Canberra."),
    dict(instruction="What is the capital of the given country?", input="Egypt", output="The capital of Egypt is Cairo."),
    dict(instruction="Summarize the following in one sentence.", input="The library opens at 9am and closes at 8pm on weekdays. On weekends it opens at 10am and closes at 6pm.", output="The library is open 9am-8pm on weekdays and 10am-6pm on weekends."),
    dict(instruction="Summarize the following in one sentence.", input="Water boils at 100 degrees Celsius at sea level. Higher altitudes lower this boiling point.", output="Water boils at 100 C at sea level but at lower temperatures at higher altitudes."),
    dict(instruction="Answer the math question.", input="What is 12 times 7?", output="12 times 7 is 84."),
    dict(instruction="Answer the math question.", input="What is 15 plus 28?", output="15 plus 28 is 43."),
    dict(instruction="Answer the math question.", input="What is 100 divided by 4?", output="100 divided by 4 is 25."),
    dict(instruction="Answer the math question.", input="What is 9 squared?", output="9 squared is 81."),
    dict(instruction="List three examples of the given category.", input="primary colors", output="Three primary colors are red, blue, and yellow."),
    dict(instruction="List three examples of the given category.", input="planets in our solar system", output="Three planets in our solar system are Earth, Mars, and Jupiter."),
    dict(instruction="List three examples of the given category.", input="programming languages", output="Three programming languages are Python, JavaScript, and Rust."),
    dict(instruction="List three examples of the given category.", input="musical instruments", output="Three musical instruments are the piano, the guitar, and the violin."),
]
# Simple split: last 4 held out for eval so we never see them during training.
train_examples = examples[:-4]
eval_examples = examples[-4:]
print(f"train={len(train_examples)}  eval={len(eval_examples)}")

## Alpaca prompt template

In [ ]:
ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides further context. "
    "Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

def format_prompt(ex):
    # Just the conditioning prompt, no target.
    return ALPACA_TEMPLATE.format(instruction=ex['instruction'], input=ex['input'])

def format_full(ex):
    # Conditioning prompt + target. The model's supervised loss covers only the target tokens.
    return format_prompt(ex) + ex['output']

print(format_full(train_examples[0]))

## Load the base model and tokenizer

In [ ]:
model_name = 'distilgpt2'  # 82M params, fits anywhere
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
base_model.config.pad_token_id = tokenizer.eos_token_id
print(f"params: {sum(p.numel() for p in base_model.parameters())/1e6:.1f}M")

## Sample BEFORE fine-tuning

distilgpt2 is a base model. It will try to continue text rather than follow instructions. Expect off-topic rambling or repetition.

In [ ]:
def generate(model, ex, max_new=60):
    model.eval()
    prompt = format_prompt(ex)
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)
    out = model.generate(
        ids, max_new_tokens=max_new,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    # Strip the prompt, return only the generated continuation.
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    model.train()
    return text

print("=== BEFORE fine-tuning ===")
for ex in eval_examples:
    print(f"[INST] {ex['instruction']} / [INP] {ex['input']}")
    print(f"[GOLD] {ex['output']}")
    print(f"[BASE] {generate(base_model, ex)}")
    print()

## Wrap with LoRA via peft

In [ ]:
# This is the production way to configure LoRA.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    # For GPT-2-style models, the attention fused projection is named "c_attn".
    target_modules=["c_attn"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## Build training batches

In [ ]:
def encode_example(ex, max_len=192):
    # Tokenize prompt and full text separately so we can mask out the prompt tokens.
    prompt = format_prompt(ex)
    full = format_full(ex) + tokenizer.eos_token  # EOS so the model learns when to stop

    full_ids = tokenizer(full, max_length=max_len, truncation=True, return_tensors='pt').input_ids[0]
    prompt_len = len(tokenizer(prompt).input_ids)

    labels = full_ids.clone()
    # Ignore prompt positions in the loss -- supervise only on the response.
    labels[:prompt_len] = -100
    return full_ids, labels

def collate(batch):
    # Right-pad all sequences to the max length in the batch.
    input_ids = [b[0] for b in batch]
    labels_list = [b[1] for b in batch]
    max_len = max(x.shape[0] for x in input_ids)
    pad_id = tokenizer.eos_token_id
    padded = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    padded_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)
    for i, (x, y) in enumerate(zip(input_ids, labels_list)):
        padded[i, :x.shape[0]] = x
        padded_labels[i, :y.shape[0]] = y
        attention_mask[i, :x.shape[0]] = 1
    return padded.to(device), attention_mask.to(device), padded_labels.to(device)

## Training loop

In [ ]:
import random
random.seed(0)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)

encoded = [encode_example(e) for e in train_examples]

n_epochs = 25
batch_size = 4
losses = []
for epoch in range(n_epochs):
    random.shuffle(encoded)
    epoch_loss = 0
    n_batches = 0
    for i in range(0, len(encoded), batch_size):
        batch = encoded[i:i+batch_size]
        input_ids, attention_mask, labels = collate(batch)
        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        opt.zero_grad(set_to_none=True)
        out.loss.backward()
        opt.step()
        epoch_loss += out.loss.item()
        n_batches += 1
    avg = epoch_loss / n_batches
    losses.append(avg)
    if epoch % 5 == 0 or epoch == n_epochs - 1:
        print(f"epoch {epoch:3d}  loss {avg:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses, marker='.')
ax.set_xlabel('epoch')
ax.set_ylabel('train loss (response tokens only)')
ax.set_title('SFT training loss')
plt.tight_layout()
plt.show()

## Sample AFTER fine-tuning

In [ ]:
print("=== AFTER LoRA SFT ===")
for ex in eval_examples:
    print(f"[INST] {ex['instruction']} / [INP] {ex['input']}")
    print(f"[GOLD] {ex['output']}")
    print(f"[SFT]  {generate(model, ex)}")
    print()

## Variation: test on an instruction *pattern* the model saw, but with new content

The dataset included templates like "What is the capital of ...?". We trained on 5 capitals; let's test one that was never in the training data to check whether the model learned the *format* or just memorized.

In [ ]:
new_examples = [
    dict(instruction="What is the capital of the given country?", input="Germany", output="The capital of Germany is Berlin."),
    dict(instruction="Answer the math question.", input="What is 6 times 9?", output="6 times 9 is 54."),
    dict(instruction="List three examples of the given category.", input="fruits", output=""),
    dict(instruction="Translate the following English to French.", input="Good night.", output=""),
]
for ex in new_examples:
    print(f"[INST] {ex['instruction']} / [INP] {ex['input']}")
    if ex['output']:
        print(f"[GOLD] {ex['output']}")
    print(f"[SFT]  {generate(model, ex)}")
    print()

## Side-by-side quality comparison

Collect base vs fine-tuned outputs for every held-out example and display them in a table. This is how production SFT quality gets reviewed: eyeballing pairs.

In [ ]:
rows = []
# Need the pre-SFT base model again for a true comparison. Reload it fresh.
eval_base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
for ex in eval_examples + new_examples:
    if not ex['output']:
        continue
    base_out = generate(eval_base, ex)
    sft_out = generate(model, ex)
    rows.append((ex['instruction'], ex['input'], ex['output'], base_out, sft_out))

for inst, inp, gold, base_out, sft_out in rows:
    print(f"instruction: {inst}")
    print(f"input:       {inp}")
    print(f"gold:        {gold}")
    print(f"base model:  {base_out[:100]}")
    print(f"SFT model:   {sft_out[:100]}")
    print('-' * 60)

## What to try next

- Scale up: download the actual `tatsu-lab/alpaca` dataset (52k examples) and train for one epoch. The jump in instruction-following quality is significant.
- Use a better base model: swap `distilgpt2` for `gpt2-medium` or a small Llama variant. Quality bump will be large.
- Add a held-out eval loop that computes loss on instructions the model never saw during training, and plot it alongside the training loss.
- Save the LoRA adapter with `model.save_pretrained('./adapter')` and load it back with `PeftModel.from_pretrained(base, './adapter')`. This is the swap-in-swap-out workflow that makes LoRA so useful in production.